In [1]:
from pathlib import Path
from collections import defaultdict

import pandas as pd

In [2]:
CLEANED_DIR = Path("../cleaned_datasets")
OUTPUT_DIR = Path("../candidate_data")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

S1_PATH = CLEANED_DIR / "cleaned_source1.tsv"
S2_PATH = CLEANED_DIR / "cleaned_source2.tsv"
S3_PATH = CLEANED_DIR / "cleaned_source3.tsv"

CANDIDATE_PATH = OUTPUT_DIR / "candidate_pairs.tsv"

CHUNK_SIZE = 100_000

print("S1:", S1_PATH)
print("S2:", S2_PATH)
print("S3:", S3_PATH)
print("Output:", CANDIDATE_PATH)

S1: ../cleaned_datasets/cleaned_source1.tsv
S2: ../cleaned_datasets/cleaned_source2.tsv
S3: ../cleaned_datasets/cleaned_source3.tsv
Output: ../candidate_data/candidate_pairs.tsv


In [3]:
for path in [S1_PATH, S2_PATH, S3_PATH]:
    print(f"{path}")
    print(f"  Exists: {path.exists()}")

    if path.exists():
        print(f"  Size: {path.stat().st_size / (1024**2):.2f} MB")

    print()

../cleaned_datasets/cleaned_source1.tsv
  Exists: True
  Size: 367.04 MB

../cleaned_datasets/cleaned_source2.tsv
  Exists: True
  Size: 854.47 MB

../cleaned_datasets/cleaned_source3.tsv
  Exists: True
  Size: 882.62 MB



In [4]:
s1 = pd.read_csv(
    S1_PATH,
    sep="\t",
    dtype=str,
    keep_default_na=False
)

print(f"S1 rows: {len(s1):,}")
print("\nColumns:")
print(s1.columns.tolist())

S1 rows: 2,206,821

Columns:
['entity_id', 'business_name', 'business_address', 'country', 'clean_business_name', 'clean_business_address', 'zip_pin', 'leading_number', 'has_non_latin_script']


In [5]:
name_index = defaultdict(list)
address_index = defaultdict(list)
name_number_index = defaultdict(list)

for row in s1.itertuples(index=False):

    entity_id = row.entity_id
    country = row.country
    name = row.clean_business_name
    address = row.clean_business_address
    leading_number = row.leading_number

    # --------------------------------------------------
    # Block 1: country + exact normalized name
    # --------------------------------------------------

    if name:
        name_index[
            (country, name)
        ].append(entity_id)

    # --------------------------------------------------
    # Block 2: country + exact normalized address
    # --------------------------------------------------

    if address:
        address_index[
            (country, address)
        ].append(entity_id)

    # --------------------------------------------------
    # Block 3: country + name + leading number
    # --------------------------------------------------

    if name and leading_number:
        name_number_index[
            (country, name, leading_number)
        ].append(entity_id)

print(f"Name blocks:        {len(name_index):,}")
print(f"Address blocks:     {len(address_index):,}")
print(f"Name + number:      {len(name_number_index):,}")

Name blocks:        1,359,749
Address blocks:     2,130,172
Name + number:      1,365,813


In [6]:
print("Example name block:")
for key, ids in list(name_index.items())[:5]:
    print(key, "->", ids[:5])

print("\nExample address block:")
for key, ids in list(address_index.items())[:5]:
    print(key, "->", ids[:5])

print("\nExample name + number block:")
for key, ids in list(name_number_index.items())[:5]:
    print(key, "->", ids[:5])

Example name block:
('US', 'orelee s barbershop') -> ['S1-925783039']
('US', 'prime money') -> ['S1-773889195']
('US', 'b retail') -> ['S1-377745466', 'S1-855081333', 'S1-810976399', 'S1-755385034']
('US', 'christ chapel') -> ['S1-133037285', 'S1-671634732', 'S1-393138020', 'S1-985657272', 'S1-560800855']
('India', 'prabhav business center') -> ['S1-755362802']

Example address block:
('US', '1795 westchester drive high point nc') -> ['S1-925783039']
('US', '17560 ellis road tahlequah ok') -> ['S1-773889195']
('US', '1712 montebello avenue phoenix az') -> ['S1-377745466']
('US', '2100 cameron drive unit apartment g dundalk md') -> ['S1-133037285']
('India', '797 lake town block a kolkata howrah west bengal') -> ['S1-755362802']

Example name + number block:
('US', 'orelee s barbershop', '1795') -> ['S1-925783039']
('US', 'prime money', '17560') -> ['S1-773889195']
('US', 'b retail', '1712') -> ['S1-377745466']
('US', 'christ chapel', '2100') -> ['S1-133037285']
('India', 'prabhav busin

In [13]:
def generate_candidates(
    source_path,
    source_name,
    output_path
):
    print(f"\n{'=' * 70}")
    print(f"Processing {source_name}")
    print(f"{'=' * 70}")

    # If the output file already exists, append.
    # Otherwise, create a new file.
    first_write = not output_path.exists()

    total_rows = 0
    total_candidates = 0

    for chunk in pd.read_csv(
        source_path,
        sep="\t",
        dtype=str,
        chunksize=CHUNK_SIZE,
        keep_default_na=False
    ):

        candidate_block_types = {}

        for row in chunk.itertuples(index=False):

            source_id = row.entity_id
            country = row.country
            name = row.clean_business_name
            address = row.clean_business_address
            leading_number = row.leading_number

            # Maps:
            # S1 ID -> set of block types
            matched_candidates = defaultdict(set)

            # --------------------------------------------------
            # BLOCK 1: exact name
            # --------------------------------------------------

            if name:
                s1_matches = name_index.get(
                    (country, name),
                    []
                )

                for s1_id in s1_matches:
                    matched_candidates[s1_id].add(
                        "name_exact"
                    )

            # --------------------------------------------------
            # BLOCK 2: exact address
            # --------------------------------------------------

            if address:
                s1_matches = address_index.get(
                    (country, address),
                    []
                )

                for s1_id in s1_matches:
                    matched_candidates[s1_id].add(
                        "address_exact"
                    )

            # --------------------------------------------------
            # BLOCK 3: exact name + leading number
            # --------------------------------------------------

            if name and leading_number:
                s1_matches = name_number_index.get(
                    (country, name, leading_number),
                    []
                )

                for s1_id in s1_matches:
                    matched_candidates[s1_id].add(
                        "name_number"
                    )

            # --------------------------------------------------
            # Store candidates
            # --------------------------------------------------

            for s1_id, block_types in matched_candidates.items():

                candidate_block_types[
                    (s1_id, source_id)
                ] = "|".join(sorted(block_types))

        # ------------------------------------------------------
        # Convert chunk candidates to DataFrame
        # ------------------------------------------------------

        if candidate_block_types:

            candidate_rows = [
                {
                    "s1_id": s1_id,
                    "source_id": source_id,
                    "source": source_name,
                    "block_types": block_types
                }
                for (s1_id, source_id), block_types
                in candidate_block_types.items()
            ]

            candidate_df = pd.DataFrame(candidate_rows)

            candidate_df.to_csv(
                output_path,
                sep="\t",
                index=False,
                mode="w" if first_write else "a",
                header=first_write
            )

            first_write = False

            total_candidates += len(candidate_df)

        total_rows += len(chunk)

        print(
            f"{source_name}: "
            f"{total_rows:,} rows processed | "
            f"{total_candidates:,} candidates"
        )

    print(
        f"\n{source_name} complete."
        f"\nRows processed: {total_rows:,}"
        f"\nCandidates:     {total_candidates:,}"
    )

In [14]:
if CANDIDATE_PATH.exists():
    CANDIDATE_PATH.unlink()
    print("Removed existing candidate_pairs.tsv")
else:
    print("No previous candidate_pairs.tsv found")

Removed existing candidate_pairs.tsv


In [15]:
generate_candidates(
    source_path=S2_PATH,
    source_name="S2",
    output_path=CANDIDATE_PATH
)


Processing S2
S2: 100,000 rows processed | 667,651 candidates
S2: 200,000 rows processed | 1,320,583 candidates
S2: 300,000 rows processed | 1,992,177 candidates
S2: 400,000 rows processed | 2,646,975 candidates
S2: 500,000 rows processed | 3,304,491 candidates
S2: 600,000 rows processed | 3,959,801 candidates
S2: 700,000 rows processed | 4,630,594 candidates
S2: 800,000 rows processed | 5,292,205 candidates
S2: 900,000 rows processed | 5,961,614 candidates
S2: 1,000,000 rows processed | 6,616,645 candidates
S2: 1,100,000 rows processed | 7,269,580 candidates
S2: 1,200,000 rows processed | 7,927,977 candidates
S2: 1,300,000 rows processed | 8,566,099 candidates
S2: 1,400,000 rows processed | 9,230,257 candidates
S2: 1,500,000 rows processed | 9,890,794 candidates
S2: 1,600,000 rows processed | 10,545,200 candidates
S2: 1,700,000 rows processed | 11,223,040 candidates
S2: 1,800,000 rows processed | 11,875,620 candidates
S2: 1,900,000 rows processed | 12,516,998 candidates
S2: 2,000,000

In [16]:
generate_candidates(
    source_path=S3_PATH,
    source_name="S3",
    output_path=CANDIDATE_PATH
)


Processing S3
S3: 100,000 rows processed | 670,840 candidates
S3: 200,000 rows processed | 1,343,836 candidates
S3: 300,000 rows processed | 2,016,574 candidates
S3: 400,000 rows processed | 2,683,864 candidates
S3: 500,000 rows processed | 3,351,544 candidates
S3: 600,000 rows processed | 4,011,676 candidates
S3: 700,000 rows processed | 4,683,269 candidates
S3: 800,000 rows processed | 5,360,288 candidates
S3: 900,000 rows processed | 6,014,966 candidates
S3: 1,000,000 rows processed | 6,692,508 candidates
S3: 1,100,000 rows processed | 7,370,867 candidates
S3: 1,200,000 rows processed | 8,035,727 candidates
S3: 1,300,000 rows processed | 8,696,240 candidates
S3: 1,400,000 rows processed | 9,365,095 candidates
S3: 1,500,000 rows processed | 10,030,679 candidates
S3: 1,600,000 rows processed | 10,706,933 candidates
S3: 1,700,000 rows processed | 11,384,910 candidates
S3: 1,800,000 rows processed | 12,057,765 candidates
S3: 1,900,000 rows processed | 12,716,629 candidates
S3: 2,000,00

In [17]:
print("Exists:", CANDIDATE_PATH.exists())

if CANDIDATE_PATH.exists():
    print(
        "Size:",
        f"{CANDIDATE_PATH.stat().st_size / (1024**2):.2f} MB"
    )

    candidates_sample = pd.read_csv(
        CANDIDATE_PATH,
        sep="\t",
        dtype=str,
        nrows=20,
        keep_default_na=False
    )

    print(candidates_sample)

Exists: True
Size: 2611.45 MB
           s1_id     source_id source block_types
0   S1-363099334  S2-163963287     S2  name_exact
1   S1-927466840  S2-163963287     S2  name_exact
2   S1-668278504  S2-163963287     S2  name_exact
3   S1-312042439  S2-163963287     S2  name_exact
4   S1-129799404  S2-163963287     S2  name_exact
5   S1-777338134  S2-163963287     S2  name_exact
6    S1-25200652  S2-163963287     S2  name_exact
7   S1-344290538  S2-163963287     S2  name_exact
8   S1-769489454  S2-163963287     S2  name_exact
9   S1-540326109  S2-163963287     S2  name_exact
10  S1-113330973  S2-163963287     S2  name_exact
11  S1-169053810  S2-163963287     S2  name_exact
12  S1-230053793  S2-163963287     S2  name_exact
13   S1-34567153  S2-163963287     S2  name_exact
14  S1-570716329  S2-163963287     S2  name_exact
15  S1-695063780  S2-163963287     S2  name_exact
16  S1-871082132  S2-163963287     S2  name_exact
17  S1-591505285  S2-163963287     S2  name_exact
18  S1-533049642  S2

In [18]:
candidate_counts = pd.read_csv(
    CANDIDATE_PATH,
    sep="\t",
    dtype=str,
    usecols=["source"]
)

print(
    candidate_counts["source"]
    .value_counts()
)

del candidate_counts

source
S3    35421238
S2    32964368
Name: count, dtype: int64


In [20]:
block_counts = pd.read_csv(
    CANDIDATE_PATH,
    sep="\t",
    dtype=str,
    usecols=["block_types"]
)

print(
    block_counts["block_types"]
    .value_counts()
)

del block_counts

block_types
name_exact                              66600602
name_exact|name_number                   1015201
address_exact                             594025
address_exact|name_exact|name_number      141939
address_exact|name_exact                   33839
Name: count, dtype: int64
